# PRNU Injection Comparison
Injects every PRNU fingerprint into a single image using both **additive** and **scaled** methods, then scores each injected image with D0_noise and reports average scores.

In [8]:
import sys
import shutil
import numpy as np
import cv2
import torch
from pathlib import Path
from PIL import Image
import torch

torch.manual_seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("../").resolve()
PRNU_DIR   = BASE_DIR / "datasets" / "prnu_fingerprints"
IMAGE_PATH = BASE_DIR / "datasets" / "genimage" / "imagenet_ai_0424_sdv5" / "train" / "ai" / "752_sdv5_00037.png"
TEMP_DIR   = BASE_DIR / "prnu_temp"
ADDITIVE_DIR = TEMP_DIR / "additive"
SCALED_DIR = TEMP_DIR / "scaled"
DYNAMIC_PSNR_DIR = TEMP_DIR / "dynamic_psnr"
D0_NOISE_PATH    = BASE_DIR / "src" / "model" / "discriminator" / "d0"
WEIGHTS_PATH     = D0_NOISE_PATH / "fc_weights.pth"
D0_PATH      = BASE_DIR / "src" / "model" / "discriminator" / "d0"
CLIP_MODEL   = BASE_DIR / "src" / "model" / "clip_model"
PCE_FRAMING_PATH = BASE_DIR / "src" / "model" / "prnu"

sys.path.insert(0, str(D0_PATH))
sys.path.insert(0, str(D0_NOISE_PATH))
sys.path.insert(0, str(PCE_FRAMING_PATH))

ADDITIVE_DIR.mkdir(parents=True, exist_ok=True)
SCALED_DIR.mkdir(parents=True, exist_ok=True)
DYNAMIC_PSNR_DIR.mkdir(parents=True, exist_ok=True)

from pce_framing import calculate_framing_success

print(f"Image     : {IMAGE_PATH}")
print(f"PRNU dir  : {PRNU_DIR}")
print(f"Temp dir  : {TEMP_DIR}")

Image     : /home/jandy/code/deepfake-it-till-you-make-it/datasets/genimage/imagenet_ai_0424_sdv5/train/ai/752_sdv5_00037.png
PRNU dir  : /home/jandy/code/deepfake-it-till-you-make-it/datasets/prnu_fingerprints
Temp dir  : /home/jandy/code/deepfake-it-till-you-make-it/prnu_temp


In [9]:
# ── Shared helpers (from both injectors) ─────────────────────────────────────
def crop_center(img_array, cropx=512, cropy=512):
    y, x = img_array.shape[:2]
    if y < cropy or x < cropx:
        if img_array.ndim == 3:
            padded = np.zeros((max(y, cropy), max(x, cropx), img_array.shape[2]), dtype=img_array.dtype)
            padded[:y, :x, :] = img_array
        else:
            padded = np.zeros((max(y, cropy), max(x, cropx)), dtype=img_array.dtype)
            padded[:y, :x] = img_array
        img_array = padded
        y, x = img_array.shape[:2]
    startx = x // 2 - cropx // 2
    starty = y // 2 - cropy // 2
    return img_array[starty:starty+cropy, startx:startx+cropx] if img_array.ndim == 2 \
           else img_array[starty:starty+cropy, startx:startx+cropx, :]

def prepare_prnu(prnu_path):
    prnu = np.load(prnu_path)
    prnu = crop_center(prnu, 512, 512)
    if prnu.ndim == 2:
        prnu = np.stack([prnu] * 3, axis=-1)
    return prnu

def load_image(image_path):
    img_bgr = cv2.imread(str(image_path))
    assert img_bgr is not None, f"Could not read image: {image_path}"
    return crop_center(img_bgr, 512, 512)

In [11]:
# ── Inject all fingerprints ───────────────────────────────────────────────────
prnu_files = sorted(PRNU_DIR.rglob("*_fingerprint.npy"))
assert prnu_files, f"No fingerprints found in {PRNU_DIR}"
print(f"Found {len(prnu_files)} fingerprints. Injecting...")

img_cropped = load_image(IMAGE_PATH)
base_name   = IMAGE_PATH.stem

for prnu_path in prnu_files:
    k = np.load(prnu_path)
    print(f"{prnu_path.stem}")
    print(f"min={k.min():.4f}, max={k.max():.4f}, std={k.std():.6f}")

import cv2
import numpy as np
from pathlib import Path

# --- THE NEW CONTROLS ---
# 40-45 dB is the forensic sweet spot. 
# > 48 dB: The noise is so weak that 8-bit rounding destroys it.
# < 35 dB: The noise looks like visible television static.
TARGET_PSNR = 45.0  
JPEG_QUALITY = 90

for prnu_path in prnu_files:
    camera_name = prnu_path.stem.replace("_fingerprint", "")
    prnu = prepare_prnu(prnu_path)

    # ========= DYNAMIC PSNR INJECTION =========
    img_float = img_cropped.astype(np.float32)
    # Calculate the raw, unscaled multiplicative noise
    raw_noise = img_float * prnu
    # Dynamic Alpha Calculation
    mse_raw = np.mean(raw_noise ** 2)
    if mse_raw <= 1e-8:
        print(f"[{camera_name}] WARNING: PRNU or image is practically blank. Skipping.")
        continue
    mse_target = (255.0 ** 2) / (10 ** (TARGET_PSNR / 10.0))
    dynamic_alpha = np.sqrt(mse_target / mse_raw)
    # Apply the perfectly scaled noise
    poisoned_float = img_float + (dynamic_alpha * raw_noise)
    
    # Strict 8-bit clipping to prevent matrix overflow
    poisoned_dynamic_psnr = np.clip(np.round(poisoned_float), 0, 255).astype(np.uint8)

    cv2.imwrite(str(DYNAMIC_PSNR_DIR / f"{base_name}_{camera_name}.png"), poisoned_dynamic_psnr)
    
    # ========= ADDITIVE INJECTION =========
    # Additive: additive in [0, 255] space
    img_float_additive  = img_cropped.astype(np.float32)
    poisoned_additive   = np.clip(img_float_additive + prnu, 0, 255).astype(np.uint8)
    cv2.imwrite(str(ADDITIVE_DIR / f"{base_name}_{camera_name}.png"), poisoned_additive)

    # ========= SCALED INJECTION =========
    # Scaled: additiveise to [0,1] then multiplicative
    img_float_scaled  = img_cropped.astype(np.float32) / 255.0
    poisoned_scaled   = np.clip(img_float_scaled * (1.0 + prnu), 0, 1.0)
    poisoned_scaled   = (poisoned_scaled * 255).astype(np.uint8)
    cv2.imwrite(str(SCALED_DIR / f"{base_name}_{camera_name}.png"), poisoned_scaled)

print(f"Done. Dynamic PSNR: {len(list(DYNAMIC_PSNR_DIR.glob('*.png')))} | Additive: {len(list(ADDITIVE_DIR.glob('*.png')))} | Scaled: {len(list(SCALED_DIR.glob('*.png')))}")

Found 35 fingerprints. Injecting...
D01_Samsung_GalaxyS3Mini_fingerprint
min=-14.3200, max=4.5555, std=0.504433
D02_Apple_iPhone4s_fingerprint
min=-92.3500, max=34.7664, std=0.482257
D03_Huawei_P9_fingerprint
min=-7.0590, max=6.7005, std=0.229065
D04_LG_D290_fingerprint
min=-15.4119, max=40.2973, std=0.566391
D05_Apple_iPhone5c_fingerprint
min=-4.6068, max=9.6606, std=0.573601
D06_Apple_iPhone6_fingerprint
min=-7.5983, max=5.5431, std=0.528482
D07_Lenovo_P70A_fingerprint
min=-3.2701, max=6.7037, std=0.410922
D08_Samsung_GalaxyTab3_fingerprint
min=-17.6380, max=14.0627, std=0.553889
D09_Apple_iPhone4_fingerprint
min=-31.4613, max=11.0823, std=1.025019
D10_Apple_iPhone4s_fingerprint
min=-8.8360, max=5.7765, std=0.738578
D11_Samsung_GalaxyS3_fingerprint
min=-5.4566, max=4.4929, std=0.365166
D12_Sony_XperiaZ1Compact_fingerprint
min=-13.8841, max=8.3129, std=1.206558
D13_Apple_iPad2_fingerprint
min=-8.2181, max=4.3375, std=0.501370
D14_Apple_iPhone5c_fingerprint
min=-11.3973, max=6.1219, st

In [12]:
# ----------get pce score
# --- Example Usage --- 
# stats = calculate_framing_success(
#     "poisoned_fakes/my_fake_D23.jpg", 
#     "datasets/prnu_fingerprints/D23_Asus_Zenfone2Laser_fingerprint.npy"
# )

def get_pce_score(image_path, fingerprint_path):
    try:
        # Assuming calculate_framing_success returns a numerical PCE score
        score = calculate_framing_success(image_path, fingerprint_path)
        return score
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return float('nan')
    
# ── Score all injected images ─────────────────────────────────────────────────
def score_folder(folder: Path, base_img_name: str, prnu_dir: Path) -> tuple[list, float]:
    scores = []
    
    # Search for both .jpg and .png
    for img_path in sorted(folder.glob("*.*")):
        if img_path.suffix.lower() not in ['.png', '.jpg', '.jpeg']:
            continue
            
        # 1. Extract the camera name from the filename
        # E.g., if filename is "fakeimg_D01_Samsung.png", and base is "fakeimg"
        # We strip the base and the underscore to get "D01_Samsung"
        filename_no_ext = img_path.stem
        camera_name = filename_no_ext.replace(f"{base_img_name}_", "")
        
        # 2. Reconstruct the exact path to the .npy fingerprint
        fingerprint_file = prnu_dir / f"{camera_name}_fingerprint.npy"
        
        if not fingerprint_file.exists():
            print(f"WARNING: Cannot find matching fingerprint for {img_path.name}. Skipping.")
            continue
            
        # 3. Calculate the actual mathematical correlation
        pce = get_pce_score(str(img_path), str(fingerprint_file))
        
        # Guard against NaN returns
        if not np.isnan(pce):
            scores.append((img_path.name, pce))
        
    if not scores:
        print(f"WARNING: No valid scores calculated in {folder}.")
        return [], 0.0
        
    avg = np.mean([score for _, score in scores])
    return scores, avg

# Note: We pass base_name (which you defined earlier as IMAGE_PATH.stem) 
# and PRNU_DIR so the function knows how to pair the images with the arrays.
scores_additive, avg_score_additive = score_folder(ADDITIVE_DIR, base_name, PRNU_DIR)
scores_scaled,   avg_score_scaled   = score_folder(SCALED_DIR, base_name, PRNU_DIR)
scores_dynamic,  avg_score_dynamic  = score_folder(DYNAMIC_PSNR_DIR, base_name, PRNU_DIR)

print("--- PCE Correlation Results ---")
print(f"Avg PCE score — Additive     : {avg_score_additive:.4f}")
print(f"Avg PCE score — Scaled       : {avg_score_scaled:.4f}")
print(f"Avg PCE score — Dynamic PSNR : {avg_score_dynamic:.4f}")

--- PCE Correlation Results ---
Avg PCE score — Additive     : 1434.5015
Avg PCE score — Scaled       : 96545.7344
Avg PCE score — Dynamic PSNR : 4845.9019


### Additive Injection (PCE: 1434.50)  
* The Math: $I_{spoofed​}=I_{clean}​+W$ (with strict rounding).  
* Took the raw noise residual and laid it flatly across the entire image matrix. Because we fixed the 8-bit truncation trap earlier, the microscopic values survived the save process.
* Interpretation: A PCE of 1400 is a pristine, textbook forensic match. It proves your spatial alignment (the 512x512 crop) is completely flawless. However, this method is physically blind. It injects the exact same noise amplitude into a pitch-black shadow as it does into a bright white cloud.  
* Verdict: It is a perfect control group, but it lacks the adversarial sophistication needed to guarantee survival. A well-trained network might easily spot the unnatural noise in the dark regions of the image.  
  
### The Scaled Injection (PCE: 96,545.73)  
* The Math: $I_{spoofed​}=I_{clean}​\times(1+K)$ forced into a $[0,1]$ clamp.   
* Took a PRNU array $(K)$ that contains massive outlier values (standard deviations of 0.5, peaks of 30+) and multiplied it directly against a normalized image. A mid-gray pixel (0.5) hit with a PRNU spike of +10 instantly became 5.5. `np.clip(..., 0, 1.0)` function then violently slammed that pixel into the 1.0 ceiling (pure white).  
* Interpretation: You didn't just break the PCE scale; you deleted the photograph. The cross-correlation algorithm isn't finding a fingerprint hidden inside an image anymore. It is simply comparing the fingerprint to itself because the semantic image data (the faces, the backgrounds) was completely clipped out of existence.  
Verdict: Absolute forensic flaw. If you feed these images to your discriminator, it won't even need to look for PRNU. It will flag them as corrupted static.  
  
### Dynamic PSNR (PCE: 4845.90)  
* Math: $I_{spoofed​}=I_{clean​}+(\alpha×I_{clean​}\times K)$, where $\alpha$ is dynamically calculated per-image to hit exactly 45 dB PSNR.  
* Forced the math to respect human visual limits while maximizing the adversarial payload. Because it is multiplicative $(I\times K)$, it naturally hides the noise in the shadows and restricts it to the highlights—mimicking true silicon physics. Because it is anchored to 45 dB, the script mathematically guarantees that the semantic image remains visually flawless to the human eye.  
* Interpretation: The payload is loud enough to deafen a machine (PCE 4800), but strictly constrained so it doesn't destroy the host image.
* Verdict: Best injection method.

In [14]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
shutil.rmtree(TEMP_DIR)
print(f"Removed temporary folder: {TEMP_DIR}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/jandy/code/deepfake-it-till-you-make-it/prnu_temp'